In [1]:
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch


c:\Users\fomka\anaconda3\envs\llm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_path = "PaddlePaddle/PaddleOCR-VL-1.5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForImageTextToText.from_pretrained(model_path,
                                                   trust_remote_code=True,
                                                   dtype=torch.bfloat16,
                                                   text_config='').to(DEVICE).eval()
processor = AutoProcessor.from_pretrained(model_path)

Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}


TypeError: PaddleOCRVLForConditionalGeneration.__init__() got an unexpected keyword argument 'text_config'

In [ ]:
from pathlib import Path

ds_path = Path('dataset')
image_path = str(ds_path / '_Distributed Cloud Computing. Big Data processing methods_ workshop.jpg')
image = Image.open(image_path).convert("RGB")
task = "ocr"
orig_w, orig_h = image.size
spotting_upscale_threshold = 1500

PROMPTS = {
    "ocr": "OCR:",
    "table": "Table Recognition:",
    "formula": "Formula Recognition:",
    "chart": "Chart Recognition:",
    "spotting": "Spotting:",
    "seal": "Seal Recognition:",
}


In [ ]:
if task == "spotting" and orig_w < spotting_upscale_threshold and orig_h < spotting_upscale_threshold:
    process_w, process_h = orig_w * 2, orig_h * 2
    try:
        resample_filter = Image.Resampling.LANCZOS
    except AttributeError:
        resample_filter = Image.LANCZOS
    image = image.resize((process_w, process_h), resample_filter)

# Set max_pixels: use 1605632 for spotting, otherwise use default ~1M pixels
max_pixels = 2048 * 28 * 28 if task == "spotting" else 1280 * 28 * 28

In [ ]:
model = AutoModelForImageTextToText.from_pretrained(model_path, torch_dtype=torch.bfloat16).to(DEVICE).eval()
processor = AutoProcessor.from_pretrained(model_path)

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPTS[task]},
        ]
    }
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    images_kwargs={"size": {"shortest_edge": processor.image_processor.min_pixels, "longest_edge": max_pixels}},
).to(model.device)

In [ ]:
inputs

In [ ]:
outputs = model.generate(**inputs, max_new_tokens=512)

In [ ]:
result = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:-1])
print(result)

In [ ]:
import json
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import io

# Попытка импорта библиотеки для работы с PDF
try:
    import fitz  # PyMuPDF
    PDF_SUPPORT = True
except ImportError:
    PDF_SUPPORT = False
    print("⚠️ Библиотека PyMuPDF не найдена. PDF-файлы будут пропущены.")
    print("   Установите: pip install PyMuPDF")

# --- Конфигурация ---
DATASET_DIR = Path('dataset')
OUTPUT_FILE = 'ocr_results.json'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
PDF_EXTENSIONS = {'.pdf'}

# --- Загрузка существующих результатов (если есть) ---
if Path(OUTPUT_FILE).exists():
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        results = json.load(f)
    processed_files = {item['file_name'] for item in results.get('data', [])}
    print(f"📂 Найден существующий файл {OUTPUT_FILE}")
    print(f"📋 Уже обработано файлов: {len(processed_files)}")
else:
    results = {
        "extraction_date": datetime.now().isoformat(),
        "data": []
    }
    processed_files = set()
    print(f"📄 Файл {OUTPUT_FILE} не найден, создаем новый")

# --- Поиск всех изображений и PDF в папке ---
all_files = [
    f for f in DATASET_DIR.iterdir()
    if f.is_file() and (f.suffix.lower() in IMAGE_EXTENSIONS or f.suffix.lower() in PDF_EXTENSIONS)
]

# --- Фильтрация: оставляем только новые файлы ---
new_files = [f for f in all_files if f.name not in processed_files][:500]
skipped_count = len(all_files) - len(new_files)

image_count = sum(1 for f in new_files if f.suffix.lower() in IMAGE_EXTENSIONS)
pdf_count = sum(1 for f in new_files if f.suffix.lower() in PDF_EXTENSIONS)

print(f"📁 Всего файлов в папке: {len(all_files)}")
print(f"⏭️ Пропущено (уже обработано): {skipped_count}")
print(f"🆕 Новые файлы для обработки: {len(new_files)}")
print(f"   ├─ Изображения: {image_count}")
print(f"   └─ PDF: {pdf_count}")

# --- Функция загрузки изображения (поддержка JPG/PNG и PDF) ---
def load_image_from_file(file_path):
    """Загружает изображение из файла (поддерживает изображения и PDF)"""
    suffix = file_path.suffix.lower()

    if suffix in IMAGE_EXTENSIONS:
        return Image.open(file_path).convert("RGB")

    elif suffix in PDF_EXTENSIONS:
        if not PDF_SUPPORT:
            raise ImportError("PyMuPDF не установлен для обработки PDF")

        # Открываем PDF и конвертируем первую страницу в изображение
        doc = fitz.open(file_path)
        page = doc[0]  # Первая страница

        # Рендерим страницу в изображение с высоким разрешением (300 DPI)
        mat = fitz.Matrix(300/72, 300/72)  # Увеличиваем разрешение для лучшего OCR
        pix = page.get_pixmap(matrix=mat)

        # Конвертируем в PIL Image
        img_data = pix.tobytes("png")
        image = Image.open(io.BytesIO(img_data)).convert("RGB")

        doc.close()
        return image

    else:
        raise ValueError(f"Неподдерживаемый формат файла: {suffix}")

# --- Обработка новых файлов ---
for file_path in tqdm(new_files, desc="Обработка сертификатов"):
    try:
        # Загрузка и подготовка изображения
        image = load_image_from_file(file_path)
        orig_w, orig_h = image.size

        # Логика масштабирования (из вашего кода)
        spotting_upscale_threshold = 1500
        if task == "spotting" and orig_w < spotting_upscale_threshold and orig_h < spotting_upscale_threshold:
            try:
                resample_filter = Image.Resampling.LANCZOS
            except AttributeError:
                resample_filter = Image.LANCZOS
            image = image.resize((orig_w * 2, orig_h * 2), resample_filter)

        # Формирование запроса к модели
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": PROMPTS[task]},
                ]
            }
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            images_kwargs={"size": {"shortest_edge": processor.image_processor.min_pixels, "longest_edge": max_pixels}},
        ).to(model.device)

        # Генерация текста
        outputs = model.generate(**inputs, max_new_tokens=512)
        result_text = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:-1])

        # Сохранение результата
        results["data"].append({
            "file_name": file_path.name,
            "result": result_text.strip()
        })

        # Сохраняем промежуточные результаты после каждого файла
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    except Exception as e:
        print(f"❌ Ошибка обработки файла {file_path.name}: {e}")
        results["data"].append({
            "file_name": file_path.name,
            "result": f"ERROR: {str(e)}"
        })
        # Сохраняем даже при ошибке
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

# --- Финальное сохранение ---
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✅ Обработка завершена!")
print(f"📊 Всего обработано файлов в этот раз: {len(new_files)}")
print(f"📈 Всего файлов в результатах: {len(results['data'])}")
print(f"💾 Результаты сохранены в {OUTPUT_FILE}")

In [ ]:
from paddleocr import PaddleOCRVL

In [ ]:
pipeline = PaddleOCRVL()

## Тест Ollama API (прямой запрос)

Прямой запрос к Ollama серверу в Docker для отладки аутентификации.

In [40]:
import requests
import json

# URL Ollama API (Docker контейнер)
OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen3-coder:480b-cloud"
API_KEY = "f9c8e4dd5802439b825c3c3052109ca3.q7ptn3hLl5PWZvwcEH7iDhee"

# Тестовый запрос
payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "Ты помощник. Отвечай кратко."},
        {"role": "user", "content": "Скажи hello"}
    ],
    "stream": False
}

# Заголовки с API ключом
headers = {
    "Content-Type": "application/json"
}

print(f"🔗 Отправка запроса к {OLLAMA_URL}/api/chat")
print(f"📝 Модель: {MODEL}")
print(f"🔑 API Key: {API_KEY[:10]}..." )
print(f"📋 Headers: {headers}")
print("-" * 50)

response = requests.post(
    f"{OLLAMA_URL}/api/chat",
    json=payload,
    headers=headers,
    timeout=120
)

print(f"📊 Status Code: {response.status_code}")
print(f"📋 Response Headers: {dict(response.headers)}")
print("-" * 50)

if response.status_code == 200:
    data = response.json()
    print(f"✅ Ответ:")
    print(data.get("message", {}).get("content", "Нет содержания"))
else:
    print(f"❌ Ошибка: {response.text}")

🔗 Отправка запроса к http://localhost:11434/api/chat
📝 Модель: qwen3-coder:480b-cloud
🔑 API Key: f9c8e4dd58...
📋 Headers: {'Content-Type': 'application/json'}
--------------------------------------------------
📊 Status Code: 401
📋 Response Headers: {'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'Content-Type': 'application/json', 'Date': 'Sat, 04 Apr 2026 14:33:41 GMT', 'Server': 'Google Frontend', 'Set-Cookie': 'aid=c5115322-c391-4669-8f6c-4ac705d81493; Path=/; Max-Age=31536000; HttpOnly; Secure; SameSite=Lax', 'Traceparent': '00-97cc0c0b8998a8ecbc44f53b026c4c4e-e49a1020eb543dfa-00', 'Via': '1.1 google', 'X-Build-Commit': '5a5dc98c56cdfa2c81cf24e2f624d9aa4bc862ed', 'X-Build-Time': '2026-04-03T17:20:33-07:00', 'X-Cloud-Trace-Context': '97cc0c0b8998a8ecbc44f53b026c4c4e/16472496320634174970', 'X-Frame-Options': 'DENY', 'X-Request-Id': '0110ecfa-d4da-46cd-bb5d-b2ab843fb3c0', 'Transfer-Encoding': 'chunked'}
--------------------------------------------------
❌ Ошибка: {"error

In [44]:
 import requests

 # Проверка версии сервера
 response = requests.get(
     "http://localhost:11434/api/version"
 )
 print(f"Version: {response.status_code}")
 print(response.text)

 # Список моделей
 response = requests.get(
     "http://localhost:11434/api/tags"
 )
 print(f"Tags: {response.status_code}")
 print(response.text)

Version: 200
{"version":"0.20.2"}
Tags: 200
{"models":[{"name":"qwen3-coder:480b-cloud","model":"qwen3-coder:480b-cloud","remote_model":"qwen3-coder:480b","remote_host":"https://ollama.com:443","modified_at":"2026-04-04T14:28:52.246661653Z","size":382,"digest":"e30e45586389a1eb8d7616d16c7b692c89b903b58456ec5f20219cd4c9737f7e","details":{"parent_model":"","format":"","family":"qwen3moe","families":["qwen3moe"],"parameter_size":"480B","quantization_level":"BF16"}}]}


## Отправка сертификата в Backend API

Отправляет `test_1.jpg` в сервис OCR через API. Возвращает `task_id` для отслеживания статуса.

In [53]:
import requests
from pathlib import Path

# URL Backend API
API_URL = "http://localhost:8000"

# Путь к тестовому файлу
image_path = Path("test_1.jpg")

# Отправка файла на обработку
with open(image_path, "rb") as f:
    response = requests.post(
        f"{API_URL}/api/certificates",
        files={"file": (image_path.name, f, "image/jpeg")},
    )

if response.status_code == 201:
    result = response.json()
    task_id = result["id"]
    print(f"✅ Задача создана!")
    print(f"📝 Task ID: {task_id}")
    print(f"📊 Статус: {result['status']}")
    print(f"💬 Сообщение: {result['message']}")
else:
    print(f"❌ Ошибка: {response.status_code}")
    print(response.json())

✅ Задача создана!
📝 Task ID: 713a3d62-8e13-40c8-8e0b-919b38131ac0
📊 Статус: pending
💬 Сообщение: Задача создана и отправлена в очередь


## Проверка статуса задачи

Выполняй эту ячейку повторно для проверки текущего статуса. Покажет прогресс обработки.

In [55]:
import requests
import json
from IPython.display import display, JSON

# URL Backend API
API_URL = "http://localhost:8000"

# Проверяем статус задачи (использует task_id из предыдущей ячейки)
if 'task_id' not in locals():
    print("⚠️ Task ID не найден. Сначала выполни ячейку отправки файла.")
else:
    response = requests.get(f"{API_URL}/api/certificates/{task_id}")
    
    if response.status_code == 200:
        data = response.json()
        status = data["status"]
        
        # Отображение статуса с эмодзи
        status_emoji = {
            "pending": "⏳",
            "ocr_processing": "🔍",
            "ocr_completed": "✅",
            "ocr_error": "❌",
            "llm_processing": "🤖",
            "completed": "🎉",
            "error": "💥",
        }
        
        print(f"📝 Task ID: {data['id']}")
        print(f"{status_emoji.get(status, '❓')} Статус: {status}")
        
        if data.get("created_at"):
            print(f"🕐 Создано: {data['created_at']}")
        
        if data.get("completed_at"):
            print(f"🏁 Завершено: {data['completed_at']}")
        
        # Если OCR завершён - показываем текст
        if data.get("raw_text"):
            print(f"\n📄 Распознанный текст ({len(data['raw_text'])} символов):")
            print("-" * 50)
            print(data['raw_text'][:500] + "..." if len(data['raw_text']) > 500 else data['raw_text'])
        
        # Если LLM завершён - показываем JSON
        if data.get("structured_data"):
            print(f"\n📊 Структурированные данные:")
            display(JSON(data['structured_data']))
        
        # Если ошибка - показываем сообщение
        if data.get("error_message"):
            print(f"\n❌ Ошибка: {data['error_message']}")

        print(data)
    else:
        print(f"❌ Ошибка: {response.status_code}")
        print(response.json())

📝 Task ID: 713a3d62-8e13-40c8-8e0b-919b38131ac0
🎉 Статус: completed
🕐 Создано: 2026-04-04T14:45:49.500178+00:00
🏁 Завершено: 2026-04-04T14:47:26.589000+00:00

📄 Распознанный текст (329 символов):
--------------------------------------------------
БЛАГОТВОРИТЕЛЬНЫЙ
ФОНД В. ПОТАНИНА

СИПЕНДИАЛЬНАЯ ПРОГРАММА
ВЛАДИМИРА ПОТАНИНА

Диплом

УЧАСТНИК Ⅱ ТУРА КОНКУРСНОГО ОТБОРА СТИПЕНДИАЛЬНОЙ ПРОГРАММЫ В. ПОТАНИНА

Купров Ауенсапр Ауенсапрович

Южно-Урацский государственной университете

(Национальной исследовательский университет)

Генеральный директор
О. И. Орачева

28. 01. 2017

📊 Структурированные данные:


<IPython.core.display.JSON object>

{'id': '713a3d62-8e13-40c8-8e0b-919b38131ac0', 'status': 'completed', 'error_message': None, 'raw_text': 'БЛАГОТВОРИТЕЛЬНЫЙ\nФОНД В. ПОТАНИНА\n\nСИПЕНДИАЛЬНАЯ ПРОГРАММА\nВЛАДИМИРА ПОТАНИНА\n\nДиплом\n\nУЧАСТНИК Ⅱ ТУРА КОНКУРСНОГО ОТБОРА СТИПЕНДИАЛЬНОЙ ПРОГРАММЫ В. ПОТАНИНА\n\nКупров Ауенсапр Ауенсапрович\n\nЮжно-Урацский государственной университете\n\n(Национальной исследовательский университет)\n\nГенеральный директор\nО. И. Орачева\n\n28. 01. 2017', 'structured_data': {'personal_data': {'full_name': 'Купров Ауенсапр Ауенсапрович', 'full_name_raw': 'Купров Ауенсапр Ауенсапрович', 'is_team': False, 'team_name': None, 'team_members': [], 'course': None, 'group': None, 'university': 'Южно-Уральский государственный университет (Национальный исследовательский университет)', 'faculty': None, 'educational_level': 'другое'}, 'document_info': {'doc_type': 'диплом', 'doc_number': None, 'doc_degree': None, 'issue_date': '2017-01-28', 'year': 2017, 'city': None, 'country': 'Россия', 'language': 